# Dengue in Vietnam: when and where does the season hit, and does rain warn us early?
Google Data Analytics Capstone, Track B. Phases: **Ask → Prepare → Process → Analyze → Share → Act**. The 503 MB OpenDengue spatial file is filtered with **DuckDB**; everything after that is **pandas**.

## 1. Ask
- **Client:** a pest-control services company (or a provincial Center for Disease Control) planning vector-control staff, insecticide stock and community campaigns for the dengue season.
- **Business task:** identify **when in the year and where** reported dengue cases in Vietnam rise, and **how strongly they follow rainfall and temperature in the preceding months**, so the client can prepare one to two months before each regional peak.
- **Stakeholders:** operations director, regional managers (North, Central coast, Central Highlands, South), finance, public-health partners.
- **Metrics:** cases per month; peak month and season length; incidence per 100,000 (if population is added); outbreak years (> mean + 2 SD); Spearman correlation of rainfall / temperature at lag 0–3 months with cases.

Full reasoning and the five guiding questions: `docs/topic_selection.md`.

## 2. Prepare
Sources, license, ROCCC and limitations: `docs/data_sources.md`. Raw files come from `python scripts/download_data.py`.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, CLEAN = ROOT / "data" / "raw", ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)
con = duckdb.connect()
pd.set_option("display.width", 120)

### 2.1 Province-level monthly cases (Vietnam)

In [2]:
prov_raw = con.execute(f'''
    SELECT adm_1_name, calendar_start_date, calendar_end_date, CAST("Year" AS INTEGER) AS year, dengue_total,
           case_definition_standardised, T_res, UUID
    FROM read_csv('{RAW / "Spatial_extract_V1_3_1.csv"}', all_varchar = true)
    WHERE upper(adm_0_name) LIKE '%VIET%' AND S_res = 'Admin1'
''').df()
prov_raw["dengue_total"] = pd.to_numeric(prov_raw["dengue_total"])
print(len(prov_raw), "rows,", prov_raw.adm_1_name.nunique(), "provinces")
print(prov_raw[["T_res", "UUID", "case_definition_standardised"]].value_counts())
print("period:", prov_raw.calendar_start_date.min(), "→", prov_raw.calendar_end_date.max())

12273 rows, 64 provinces
T_res  UUID                           case_definition_standardised
Month  TYCHO-ALL-19242017-SV_DF01-00  Total                           12273
Name: count, dtype: int64
period: 1994-02-01 → 2010-12-31


In [3]:
checks = {
    "negative counts": int((prov_raw.dengue_total < 0).sum()),
    "missing counts": int(prov_raw.dengue_total.isna().sum()),
    "duplicate province-months": int(prov_raw.duplicated(["adm_1_name", "calendar_start_date"]).sum()),
    "rows not one calendar month": int((pd.to_datetime(prov_raw.calendar_start_date).dt.to_period("M")
                                         != pd.to_datetime(prov_raw.calendar_end_date).dt.to_period("M")).sum()),
}
checks

{'negative counts': 0,
 'missing counts': 0,
 'duplicate province-months': 0,
 'rows not one calendar month': 0}

In [4]:
coverage = (prov_raw.groupby("adm_1_name")
            .agg(months=("calendar_start_date", "nunique"), first=("calendar_start_date", "min"),
                 last=("calendar_start_date", "max"), cases=("dengue_total", "sum"))
            .sort_values("cases", ascending=False))
print(coverage.months.value_counts().sort_index(ascending=False))
coverage.head(12)

months
203    30
202    13
201     5
200     3
191     1
168     7
167     1
166     1
84      3
Name: count, dtype: int64


,months,first,last,cases
adm_1_name,,,,
HO CHI MINH CITY,203,1994-02-01,2010-12-01,115304
TIEN GIANG,203,1994-02-01,2010-12-01,83373
BEN TRE,201,1994-02-01,2010-12-01,82532
DONG THAP,203,1994-02-01,2010-12-01,72717
BINH THUAN,200,1994-02-01,2010-12-01,68206
CA MAU,202,1994-02-01,2010-12-01,68024
AN GIANG,203,1994-02-01,2010-12-01,67580
KIEN GIANG,203,1994-02-01,2010-12-01,66764
DONG NAI,203,1994-02-01,2010-12-01,63879


### 2.2 National series (all resolutions) and cross-check

In [5]:
nat = pd.read_csv(RAW / "National_extract_V1_3_1.csv", low_memory=False)
nat_vn = nat[nat.adm_0_name.str.upper().str.contains("VIET")].copy()
print(nat_vn.groupby("T_res").Year.agg(["min", "max", "count"]))
cross = pd.concat([
    prov_raw.groupby("year").dengue_total.sum().rename("sum_of_provinces"),
    nat_vn[nat_vn.T_res == "Month"].groupby("Year").dengue_total.sum().rename("national_monthly"),
    nat_vn[nat_vn.T_res == "Year"].set_index("Year").dengue_total.rename("national_yearly"),
], axis=1).loc[1994:2010]
cross

        min   max  count
T_res                   
Month  1995  2025    183
Week   2016  2023     93
Year   1960  2022     41


,sum_of_provinces,national_monthly,national_yearly
1994,42828.0,NaN,44944.0
1995,80447.0,80447.0,NaN
1996,89963.0,89963.0,NaN
1997,107188.0,NaN,108000.0
1998,234920.0,234920.0,NaN
1999,35868.0,35868.0,NaN
2000,25269.0,25269.0,NaN
2001,42013.0,NaN,42878.0
2002,31608.0,NaN,31760.0
2003,49751.0,49751.0,NaN


### 2.3 Weather (NASA POWER, 11 provincial capitals)

In [6]:
def read_power(path):
    lines = path.read_text().splitlines()
    start = lines.index("-END HEADER-") + 1
    from io import StringIO
    w = pd.read_csv(StringIO("\n".join(lines[start:])))
    w.insert(0, "file", path.stem)
    return w

weather_wide = pd.concat([read_power(p) for p in sorted((RAW / "weather").glob("*.csv"))], ignore_index=True)
print(weather_wide.shape, weather_wide.file.nunique(), "locations,", weather_wide.YEAR.min(), "→", weather_wide.YEAR.max())
print("missing (-999):", int((weather_wide.select_dtypes("number") == -999).sum().sum()))
weather_wide.head(3)

(1023, 16) 11 locations, 1994 → 2024
missing (-999): 0


,file,PARAMETER,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANN
0,an-giang,PRECTOTCORR,1994,0.34,0.00,0.94,2.11,5.27,8.88,5.12,7.47,6.82,4.37,0.95,1.24,3.64
1,an-giang,PRECTOTCORR,1995,0.34,0.00,0.43,1.06,6.45,7.91,9.07,8.52,9.27,9.88,3.26,2.02,4.88
2,an-giang,PRECTOTCORR,1996,0.78,0.13,0.13,3.96,8.40,6.64,9.74,7.22,8.10,10.82,6.75,1.31,5.35


### Prepare: what the checks show
- **Scope:** province-level monthly data for 64 provinces, 1994-02 → 2010-12 (one source, Project Tycho via OpenDengue), with no negative, missing or duplicate rows. This supports seasonality, regional comparison and the weather-lag analysis at province level.
- **Consistency:** the sum of provinces matches the national monthly series exactly in every year where both exist, and is 0.1–2.0% below the national annual figure in 1997, 2001, 2002 and 2010 (4.7% in 1994, whose January is missing at province level).
- **Current:** province data stops in 2010; the national monthly series for 2024–2025 is used to check whether the seasonal timing still holds.
- **Weather:** 11 provincial capitals, 1994–2024, no missing values.

## 3. Process
Log every step with row counts in `docs/changelog.md`.
- Parse `calendar_start_date` → `month`; add region (North, Central coast, Central Highlands, South) to each province.
- Weather wide → long (`location, year, month, T2M, PRECTOTCORR, RH2M`); precipitation mm/day → mm/month.
- Join cases and weather on province + year + month; add `rain_lag1..3`, `temp_lag1..3`.
- National annual series: combine yearly, monthly and weekly sources without double counting; do not fill gaps with 0.

## 4. Analyze
1. Long-term trend and outbreak years (national).
2. Seasonality: mean cases by month; heatmap year × month; peak month and season length by region.
3. Province ranking (cases; incidence if population is added).
4. Climate lead time: Spearman correlation of rainfall / temperature at lag 0–3 with cases, per province.
5. (Advanced) STL decomposition: is the season stable across years?

## 5. Share
Charts whose titles state the finding, saved to `figures/`; Tableau dashboard (heatmap, map, lag chart); 14–16 slides for ~30 minutes.

## 6. Act
**Top high-level insights:** …

**Additional deliverables for further exploration:** …